# Tutorial about fluopy - distributed simulation

Here we outline parallel computing with fluopy.

Install [ray](https://docs.ray.io) for distributing multiple simulations on different cores or machines.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import ray

import fluopy
import fluopy.fluorophores as fl
import fluopy.simulation as si
import fluopy.transitions as tr

In [ ]:
fluopy.__version__

## Define a fluorophore and transition set

In [ ]:
fluorophore = fl.Fluorophore(name="cy5_dna", position=[0, 0])
fluorophore_system = fl.FluorophoreSystem(fluorophores=[fluorophore])

In [ ]:
transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=2,
    wavelength=640,
    bleaching=True,
    energy_transfer=False,
    dstorm=False,
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set.finalize()

In [ ]:
transition_set.transition_df

## Run a simulation in parallel using ray

In [ ]:
ray.init()

In [ ]:
%%time


@ray.remote
def worker(transition_set, seed):
    simulation = si.Simulation(transition_set)
    simulation.run(start_at=None, size=1e6, end_time=None, seed=seed)
    return simulation


n_processes = 10
ss = np.random.SeedSequence()
child_seeds = ss.spawn(n_processes)

futures = [
    worker.remote(transition_set=transition_set, seed=seed) for seed in child_seeds
]
simulations = ray.get(futures)

In [ ]:
ray.shutdown()  # might be needed to reinitialize ray if executed multiple times

### Extract and present data

In [ ]:
final_times = [simulation_.time_series[-1] for simulation_ in simulations]
len(final_times)

In [ ]:
plt.hist(final_times)
plt.gca().set(
    title=f"The mean bleaching time of {len(simulations)} simulations is {np.mean(final_times):0.2f} s",
    xlabel="bleaching time / s",
    ylabel="counts",
);